In [1]:
from statistics import linear_regression

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix as cm

train_data = pd.read_csv('train.csv')
train_data.head()

,attrition,training_spend,education_level,job_title,gender_male,performance_rating,boss_rating_avg,salary,overtime_per_week,traveldays_per_year,lastpromotion_months,salary_difference
0,False,1860,BSc,Developer,True,4,3.2,5972,7,8,23,-75.771429
1,False,2390,MSc or MBA,Junior designer,True,4,4.3,4262,7,16,24,32.785714
2,False,2018,BSc,Associate,True,4,2.0,5474,3,28,4,35.230769
3,False,2429,MSc or MBA,Junior developer,True,3,3.5,3967,3,8,5,45.142361
4,False,1684,BSc,Associate,True,2,1.4,4962,7,9,15,-121.344828


Encoding the categorical data with get_dummies and Booleans (attrition & gender_male) to numeric (0=False, 1=True)

In [2]:
train_data_encoded = pd.get_dummies(train_data, columns = ['education_level', 'job_title'], drop_first= True, dtype = int)

train_data_encoded['attrition'] = train_data_encoded['attrition'].astype(int)
train_data_encoded['gender_male'] = train_data_encoded['gender_male'].astype(int)

train_data_encoded.head()

,attrition,training_spend,gender_male,performance_rating,boss_rating_avg,salary,overtime_per_week,traveldays_per_year,lastpromotion_months,salary_difference,...,job_title_Consultant,job_title_Designer,job_title_Developer,job_title_Junior designer,job_title_Junior developer,job_title_Lead designer,job_title_Lead developer,job_title_Principal consultant,job_title_Senior designer,job_title_Senior developer
0,0,1860,1,4,3.2,5972,7,8,23,-75.771429,...,0,0,1,0,0,0,0,0,0,0
1,0,2390,1,4,4.3,4262,7,16,24,32.785714,...,0,0,0,1,0,0,0,0,0,0
2,0,2018,1,4,2.0,5474,3,28,4,35.230769,...,0,0,0,0,0,0,0,0,0,0
3,0,2429,1,3,3.5,3967,3,8,5,45.142361,...,0,0,0,0,1,0,0,0,0,0
4,0,1684,1,2,1.4,4962,7,9,15,-121.344828,...,0,0,0,0,0,0,0,0,0,0


Running the first logistic regression

In [3]:
y = train_data_encoded['attrition']
X = train_data_encoded.drop('attrition', axis = 1)

X = sm.add_constant(X)
log_res_model1 = sm.Logit(y, X)
result1 = log_res_model1.fit()

print(result1.summary())

Optimization terminated successfully.
         Current function value: 0.432551
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              attrition   No. Observations:                 1731
Model:                          Logit   Df Residuals:                     1708
Method:                           MLE   Df Model:                           22
Date:                Sun, 15 Mar 2026   Pseudo R-squ.:                  0.1325
Time:                        10:19:22   Log-Likelihood:                -748.75
converged:                       True   LL-Null:                       -863.14
Covariance Type:            nonrobust   LLR p-value:                 2.420e-36
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                              2.6580      4.738      0.561     

Dropping salary and running the logistic regression again

In [4]:
train_data_encoded = train_data_encoded.drop('salary', axis = 1)

In [5]:
y = train_data_encoded['attrition']
X = train_data_encoded.drop('attrition', axis = 1)

X = sm.add_constant(X)
log_res_model2 = sm.Logit(y, X)
result2 = log_res_model2.fit()

print(result2.summary())

Optimization terminated successfully.
         Current function value: 0.432676
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              attrition   No. Observations:                 1731
Model:                          Logit   Df Residuals:                     1709
Method:                           MLE   Df Model:                           21
Date:                Sun, 15 Mar 2026   Pseudo R-squ.:                  0.1323
Time:                        10:19:22   Log-Likelihood:                -748.96
converged:                       True   LL-Null:                       -863.14
Covariance Type:            nonrobust   LLR p-value:                 8.797e-37
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                             -0.4159      0.673     -0.618     

Odds ratios for all variables

In [6]:
np.exp(result2.params)

const                             0.659763
training_spend                    0.999147
gender_male                       1.115742
performance_rating                0.686540
boss_rating_avg                   0.697656
overtime_per_week                 1.117353
traveldays_per_year               1.027013
lastpromotion_months              1.055889
salary_difference                 0.999424
education_level_DSc or PhD        0.673987
education_level_MSc or MBA        1.370539
job_title_Business analyst        0.714128
job_title_Consultant              0.621502
job_title_Designer                2.218784
job_title_Developer               2.034809
job_title_Junior designer         1.930302
job_title_Junior developer        1.884599
job_title_Lead designer           1.170611
job_title_Lead developer          1.477721
job_title_Principal consultant    0.551539
job_title_Senior designer         0.964084
job_title_Senior developer        1.005736
dtype: float64

decrease in overtime of 3 hours leads to change of :

In [7]:
base = result2.params['overtime_per_week']
change = -3

multiplier = np.exp(base * change)

multiplier -1

np.float64(-0.2831485295712308)

New attrition risk if travel days would be decreased by 15 days a year

In [8]:
#We get the base case coefficient
base2 = result2.params['traveldays_per_year']

odds = 0.25
change2 = -15

new_odds = odds * np.exp(base2 * change2)
new_prob = new_odds / (1 + new_odds)


print(new_prob)

0.143550101341541


Confusion matrix and missed attritions & prediction accuracy

In [9]:
base3 = result2.params['training_spend']
change3 = 200

multiplier2 = np.exp(base3 * change3)

multiplier2 -1

np.float64(-0.15693215478520828)

In [10]:
# resycling some previous values as they do not change
new_odds2 = odds * np.exp(base3 * change3)
new_prob2 = new_odds2 / (1 + new_odds2)


print(new_prob2)

0.17407723206846823


Now test data!

In [11]:
test_data = pd.read_csv('test.csv')

test_encoded = pd.get_dummies(test_data, columns = ['education_level', 'job_title'], drop_first= True, dtype = int)

test_encoded['attrition'] = test_encoded['attrition'].astype(int)
test_encoded['gender_male'] = test_encoded['gender_male'].astype(int)

test_encoded = test_encoded.drop("salary", axis=1)

test_encoded.head()

,attrition,training_spend,gender_male,performance_rating,boss_rating_avg,overtime_per_week,traveldays_per_year,lastpromotion_months,salary_difference,education_level_DSc or PhD,...,job_title_Consultant,job_title_Designer,job_title_Developer,job_title_Junior designer,job_title_Junior developer,job_title_Lead designer,job_title_Lead developer,job_title_Principal consultant,job_title_Senior designer,job_title_Senior developer
0,0,1942,1,2,3.1,0,19,17,-96.767677,0,...,0,0,0,0,0,0,0,0,0,0
1,0,2032,1,4,3.1,2,20,10,-17.976378,0,...,0,0,0,0,1,0,0,0,0,0
2,1,2115,0,1,2.6,10,16,23,-191.857639,0,...,0,0,0,0,1,0,0,0,0,0
3,0,2028,0,3,3.3,5,27,16,33.030888,0,...,0,0,0,1,0,0,0,0,0,0
4,0,1946,0,3,3.2,2,24,15,19.655172,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
y_test = test_encoded["attrition"]
X_test = test_encoded.drop("attrition", axis=1)

# align columns with training model
X_test = X_test.reindex(columns=X.columns.drop("const"), fill_value=0)
X_test = sm.add_constant(X_test)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

y_prob_test = result2.predict(X_test)
y_pred_test = (y_prob_test > 0.5).astype(int)

matrix =cm(y_test, y_pred_test)

print(matrix)
print(matrix[1,1])

[[340  13]
 [ 55  11]]
11


In [13]:
accuracy = (matrix[0,0] + matrix[1,1]) / matrix.sum()
accuracy

np.float64(0.837708830548926)